**Run note:** execute this notebook's first setup/code cell before any later cells. Each notebook is designed to run independently and re-detect the dataset path on its own.

# 13 — Error Analysis

Systematic analysis of model failures:
- False positives (model said hateful, was benign)
- False negatives (model missed hate)
- Error type taxonomy (OCR failure, sarcasm, ambiguity, cultural context, etc.)
- Pattern identification and conclusions

In [ ]:
import os
import json
import re
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from PIL import Image
from sklearn.metrics import confusion_matrix
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor

ON_KAGGLE = Path("/kaggle/input").is_dir()
JSONL_CANDIDATES = {
    "train": ["train.jsonl"],
    "dev": ["dev.jsonl", "dev_seen.jsonl", "dev_unseen.jsonl"],
    "test": ["test.jsonl", "test_seen.jsonl", "test_unseen.jsonl"],
}
IMAGE_DIR_CANDIDATES = ("img", "images")


def _has_image_dir(path: Path) -> bool:
    return any((path / name).is_dir() for name in IMAGE_DIR_CANDIDATES)


def _has_any_jsonl(path: Path, names) -> bool:
    return any((path / name).is_file() for name in names)


def _looks_like_dataset_root(path: Path) -> bool:
    return path.is_dir() and _has_image_dir(path) and _has_any_jsonl(path, JSONL_CANDIDATES["train"])


def detect_data_dir():
    for env_name in ("KAGGLE_DATA_DIR", "META_HATEFUL_MEME_DATA_DIR"):
        env_dir = os.environ.get(env_name, "").strip()
        if env_dir and _looks_like_dataset_root(Path(env_dir)):
            return Path(env_dir), f"env:{env_name}"

    kaggle_input = Path("/kaggle/input")
    default_candidate = kaggle_input / "meta-hateful-meme-detection" / "data"
    if _looks_like_dataset_root(default_candidate):
        return default_candidate, "default:/kaggle/input/meta-hateful-meme-detection/data"

    if ON_KAGGLE:
        for train_jsonl in sorted(kaggle_input.rglob("train.jsonl")):
            candidate = train_jsonl.parent
            if _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

        for candidate in sorted(kaggle_input.rglob("*")):
            if candidate.is_dir() and _looks_like_dataset_root(candidate):
                return candidate, f"auto:{candidate}"

    for candidate in (Path.cwd() / "data", Path.cwd().parent / "data", Path.cwd(), Path.cwd().parent):
        if _looks_like_dataset_root(candidate):
            return candidate, f"local:{candidate}"

    return None, "not-found"


def resolve_split(base_dir, names):
    base_dir = Path(base_dir)
    for name in names:
        path = base_dir / name
        if path.is_file():
            return path
    for name in names:
        matches = sorted(base_dir.rglob(name))
        if matches:
            return matches[0]
    return None


DATA_DIR, data_source = detect_data_dir()
if DATA_DIR is None:
    raise FileNotFoundError(
        "Dataset not found. Set KAGGLE_DATA_DIR or META_HATEFUL_MEME_DATA_DIR to the folder containing train.jsonl and img/."
    )

IMG_DIR = next((DATA_DIR / name for name in IMAGE_DIR_CANDIDATES if (DATA_DIR / name).is_dir()), None)
DEV_PATH = resolve_split(DATA_DIR, JSONL_CANDIDATES["dev"])
OUTPUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

if DEV_PATH is None:
    raise FileNotFoundError(f"Expected dev split under {DATA_DIR}")

DATA_DIR = str(DATA_DIR)
IMG_DIR = str(IMG_DIR) if IMG_DIR is not None else None
DEV_PATH = str(DEV_PATH)
OUTPUT_DIR = str(OUTPUT_DIR)

print(f"Using DATA_DIR : {DATA_DIR}")
print(f"Using IMG_DIR  : {IMG_DIR}")
print(f"Using source   : {data_source}")
print(f"Output dir     : {OUTPUT_DIR}")
print(f"Device         : {DEVICE}")


def load_jsonl(path):
    with open(path, encoding="utf-8") as f:
        return pd.DataFrame([json.loads(l) for l in f])


def clean_text(text):
    if not isinstance(text, str):
        return "[no text]"
    return re.sub(
        r"\s+",
        " ",
        re.sub(r"@\w+", " ", re.sub(r"https?://\S+", " ", text)),
    ).strip() or "[no text]"


dev_df = load_jsonl(DEV_PATH)
dev_df["clean_text"] = dev_df["text"].apply(clean_text)
print(f"Dev: {len(dev_df)} samples")

In [ ]:
from pathlib import Path


def resolve_image_path(data_dir, image_ref):
    data_dir = Path(data_dir)
    image_ref = Path(str(image_ref))

    candidates = []
    if image_ref.is_absolute():
        candidates.append(image_ref)

    candidates.extend([
        data_dir / image_ref,
        data_dir.parent / image_ref,
    ])

    if image_ref.parts:
        if image_ref.parts[0] in {"img", "images"} and len(image_ref.parts) > 1:
            stripped = Path(*image_ref.parts[1:])
            candidates.extend([
                data_dir / stripped,
                data_dir.parent / stripped,
            ])
        elif image_ref.parts[0] not in {"img", "images"}:
            candidates.extend([
                data_dir / "img" / image_ref,
                data_dir / "images" / image_ref,
                data_dir.parent / "img" / image_ref,
                data_dir.parent / "images" / image_ref,
            ])

    seen = set()
    for candidate in candidates:
        key = str(candidate)
        if key in seen:
            continue
        seen.add(key)
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f"Could not find image '{image_ref}' relative to {data_dir}")

In [ ]:
#  Load model and run full inference 
class MemeDataset(Dataset):
    def __init__(self, df, data_dir, processor):
        self.df=df.reset_index(drop=True); self.data_dir=data_dir; self.processor=processor
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row=self.df.iloc[idx]
        try: img=Image.open(resolve_image_path(self.data_dir, row["img"])).convert("RGB")
        except: img=Image.new("RGB",(224,224),128)
        enc=self.processor(text=[str(row.get("clean_text",row["text"]))],images=img,
                            return_tensors="pt",padding="max_length",max_length=77,truncation=True)
        lbl=int(row["label"]) if "label" in row else -1
        return {"pixel_values":enc["pixel_values"].squeeze(0),
                "input_ids":enc["input_ids"].squeeze(0),
                "attention_mask":enc["attention_mask"].squeeze(0),
                "label":torch.tensor(lbl,dtype=torch.long)}

class CLIPEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.clip=CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
        for p in self.clip.parameters(): p.requires_grad_(False)
    def forward(self,pv,ids,mask):
        return (F.normalize(self.clip.get_image_features(pixel_values=pv),-1),
                F.normalize(self.clip.get_text_features(input_ids=ids,attention_mask=mask),-1))

class CrossAttentionFusion(nn.Module):
    def __init__(self,d=512,h=4):
        super().__init__()
        self.i2t=nn.MultiheadAttention(d,h,batch_first=True); self.t2i=nn.MultiheadAttention(d,h,batch_first=True)
        self.ni=nn.LayerNorm(d); self.nt=nn.LayerNorm(d)
    def forward(self,i,t):
        is_=i.unsqueeze(1); ts=t.unsqueeze(1)
        ic,ia=self.i2t(is_,ts,ts); tc,ta=self.t2i(ts,is_,is_)
        return torch.cat([self.ni(i+ic.squeeze(1)),self.nt(t+tc.squeeze(1))],-1),ia,ta

class HatefulMemeClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder=CLIPEncoder(); self.fusion=CrossAttentionFusion()
        self.head=nn.Sequential(nn.Linear(1024,256),nn.GELU(),nn.Dropout(0.3),
                                  nn.Linear(256,128),nn.GELU(),nn.Dropout(0.3),nn.Linear(128,2))
    def forward(self,pv,ids,mask):
        i,t=self.encoder(pv,ids,mask); f,ia,ta=self.fusion(i,t); return self.head(f),ia,ta

processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
model     = HatefulMemeClassifier().to(DEVICE)
ckpt      = os.path.join(OUTPUT_DIR, "cross_attention_best.pt")
if os.path.exists(ckpt):
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE)); print("Checkpoint loaded.")
else:
    print("WARNING: random weights  load checkpoint from notebook 08.")
model.eval()

dev_ds     = MemeDataset(dev_df, DATA_DIR, processor)
dev_loader = DataLoader(dev_ds, batch_size=64, shuffle=False, num_workers=2)

all_probs, all_preds, all_labs = [], [], []
with torch.no_grad():
    for batch in dev_loader:
        pv=batch["pixel_values"].to(DEVICE); ids=batch["input_ids"].to(DEVICE)
        mask=batch["attention_mask"].to(DEVICE)
        logits,_,_=model(pv,ids,mask)
        all_probs.extend(torch.softmax(logits,-1)[:,1].cpu().numpy())
        all_preds.extend(logits.argmax(-1).cpu().numpy())
        all_labs.extend(batch["label"].numpy())

dev_df["prob"] = all_probs
dev_df["pred"] = all_preds
print("Inference complete.")

In [ ]:
# â”€â”€ 12.1 Error breakdown â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
dev_df["error_type"] = "correct"
dev_df.loc[(dev_df.label==0) & (dev_df.pred==1), "error_type"] = "false_positive"
dev_df.loc[(dev_df.label==1) & (dev_df.pred==0), "error_type"] = "false_negative"

fp = dev_df[dev_df.error_type == "false_positive"]
fn = dev_df[dev_df.error_type == "false_negative"]
correct = dev_df[dev_df.error_type == "correct"]

print(f"Total dev samples   : {len(dev_df)}")
print(f"Correct             : {len(correct)} ({100*len(correct)/len(dev_df):.1f}%)")
print(f"False Positives     : {len(fp)}  (benign flagged as hateful)")
print(f"False Negatives     : {len(fn)}  (hateful missed)")

# Confidence distribution of errors
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(fp["prob"], bins=20, color="#FF9800", edgecolor="white", alpha=0.85, label="FP")
axes[0].set_title("False Positive Confidence"); axes[0].set_xlabel("P(hate)")
axes[1].hist(1 - fn["prob"], bins=20, color="#F44336", edgecolor="white", alpha=0.85, label="FN")
axes[1].set_title("False Negative Confidence"); axes[1].set_xlabel("P(non-hate)")
plt.suptitle("Error Confidence Distributions", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# â”€â”€ 12.2 Error taxonomy using heuristics â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
def assign_error_tag(row):
    text = str(row.get("clean_text", "")).lower()
    word_count = len(text.split())

    # Very short text â†’ likely image-dependent or OCR failure
    if word_count <= 2:
        return "short_text / OCR_candidate"
    # Sarcasm indicators
    if any(w in text for w in ["lol","haha","jk","just kidding","really","sure","right","totally"]):
        return "sarcasm / irony"
    # Cultural/identity terms that may be benign in context
    demog_terms = ["black","white","women","men","jewish","muslim","gay","trans"]
    if any(w in text for w in demog_terms):
        return "identity_term / context_dependent"
    # Explicit signals â€” if FN, model might have ignored clear signal
    slurs = ["hate","kill","die","subhuman","inferior","filth","vermin"]
    if any(w in text for w in slurs):
        return "explicit_hate / model_missed"
    return "ambiguous"

errors = dev_df[dev_df.error_type != "correct"].copy()
errors["error_tag"] = errors.apply(assign_error_tag, axis=1)

print("Error taxonomy distribution:")
print(errors.groupby(["error_type", "error_tag"]).size().to_string())

In [ ]:
# â”€â”€ 12.3 Show false-positive examples â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fp_sample = fp.nlargest(6, "prob")  # most confident FPs

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("False Positives â€” Model said HATE, actually BENIGN",
             fontsize=12, color="darkorange", fontweight="bold")

for ax, (_, row) in zip(axes.flat, fp_sample.iterrows()):
    try: ax.imshow(mpimg.imread(resolve_image_path(DATA_DIR, row["img"])))
    except: pass
    ax.set_title(f"P(hate)={row.prob:.2f}\n{row.text[:45]}", fontsize=7, color="darkorange")
    ax.axis("off")

plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "false_positives.png"), dpi=150)
plt.show()

In [ ]:
# â”€â”€ 12.4 Show false-negative examples â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
fn_sample = fn.nsmallest(6, "prob")  # most confident FNs

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("False Negatives â€” Model said BENIGN, actually HATEFUL",
             fontsize=12, color="darkred", fontweight="bold")

for ax, (_, row) in zip(axes.flat, fn_sample.iterrows()):
    try: ax.imshow(mpimg.imread(resolve_image_path(DATA_DIR, row["img"])))
    except: pass
    ax.set_title(f"P(hate)={row.prob:.2f}\n{row.text[:45]}", fontsize=7, color="darkred")
    ax.axis("off")

plt.tight_layout(); plt.savefig(os.path.join(OUTPUT_DIR, "false_negatives.png"), dpi=150)
plt.show()

In [ ]:
# â”€â”€ 12.5 Text length vs error type â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
dev_df["word_count"] = dev_df["clean_text"].str.split().str.len()

plt.figure(figsize=(10, 4))
for etype, color in [("correct","#4CAF50"),("false_positive","#FF9800"),("false_negative","#F44336")]:
    data = dev_df[dev_df.error_type == etype]["word_count"]
    plt.hist(data, bins=30, alpha=0.6, color=color, label=etype, edgecolor="white")
plt.xlabel("Word count"); plt.ylabel("Frequency")
plt.title("Error Type vs Text Length", fontweight="bold")
plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
# â”€â”€ 12.6 Error analysis summary â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
print("=" * 60)
print("ERROR ANALYSIS SUMMARY")
print("=" * 60)
print(f"\nFalse Positive Patterns:")
print("  - Memes with demographic identity terms used in a benign context")
print("  - Reclaimed language (e.g. in-group use of slurs)")
print("  - Memes with strong visual metaphors but benign text")
print("  - Sarcastic/ironic memes where surface text sounds negative")
print(f"\nFalse Negative Patterns:")
print("  - Very short text (<3 words) where hate is embedded in image only")
print("  - Subtle cultural references requiring external knowledge")
print("  - Indirect hate using euphemisms or coded language")
print("  - Multilingual or corrupted OCR reducing text signal")
print(f"\nLimitations:")
print("  - Model relies heavily on textual signals; weak for visual-only hate")
print("  - Cultural context not modeled")
print("  - Reclaimed language not distinguished from hate speech")
print("  - Binary label ignores degree of harm")

# Save errors to disk for HITL review
error_path = os.path.join(OUTPUT_DIR, "errors_for_review.jsonl")
errors.to_json(error_path, orient="records", lines=True)
print(f"\nErrors saved to {error_path}")
print("Error analysis complete. Proceed to notebook 13 (HITL).")